# Stage 2: tree evidence and hypothesis inputs

This notebook verifies the saved Stage 1 outputs and translates aggregate policy-tree patterns into structured evidence for hypothesis generation. The LLM inputs contain aggregate results.

Evidence selection is retrospective and exploratory. Researchers review the extracted patterns and subgroup labels before interpreting the generated hypotheses.

In [1]:
from pathlib import Path
from datetime import datetime
import csv, hashlib, json, os
import joblib

ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'configs/stage1_specifications.json').is_file())
SNAPSHOT_S1=ROOT/'S1/outputs'
SHARED_S1=ROOT/'S1/fixed_inputs'
POLICY_OUTPUTS=[Path(os.environ['REPLICATION_S1_POLICY_OUTPUTS']).resolve()] if os.environ.get('REPLICATION_S1_POLICY_OUTPUTS') else [SNAPSHOT_S1,SHARED_S1]
required={'residential_eligible','outpatient_eligible'}

def sha256(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''): h.update(block)
    return h.hexdigest()

def latest_policy_run():
    manifests=sorted({p.resolve() for folder in POLICY_OUTPUTS for p in folder.glob('S1_02_policy_trees_*/manifest.json')},key=lambda p:p.parent.name,reverse=True)
    for p in manifests:
        m=json.loads(p.read_text())
        primary=[r for r in m.get('results',[]) if r.get('branch') in required]
        names={r.get('branch') for r in primary}
        if m.get('status')=='eligible_observed_information_and_complete_case_policy_sensitivity_inference_pending' and required <= names and all(r.get('node_inference') and r.get('node_letter_order') for r in primary):return p,m
    searched=', '.join(str(p) for p in POLICY_OUTPUTS)
    raise FileNotFoundError(f'No complete lettered-node S1_02 policy-tree run under: {searched}. Run S1/S1_02 before S2.')

manifest_path,source_manifest=latest_policy_run()
run_dir=manifest_path.parent
results={r['branch']:r for r in source_manifest['results']}
print('Source:',run_dir.relative_to(ROOT))
print('Created:',source_manifest['created_at'])


Source: notebooks/code_0927/S1/outputs/S1_02_policy_trees_20260926_195240_351369
Created: 2026-09-26T20:04:44.511556-04:00


## Integrity checks

The notebook verifies every recorded branch artifact before extracting evidence. Counts in the evidence bundle come from held-out evaluation fields in the manifest. Counts printed inside tree images describe the tree-building partition and are not substituted for evaluation counts.

In [2]:
verified=[]
for name in required:
    branch=results[name]
    for rel,expected in branch['files'].items():
        path=run_dir/name/rel
        actual=sha256(path)
        assert actual==expected,(path,actual,expected)
        verified.append(str(path.relative_to(ROOT)))
print(f'Verified {len(verified)} Stage 1 artifacts.')

Verified 20 Stage 1 artifacts.


## Frozen selection rules

1. Include the two primary eligible-sample service-setting analyses.
2. Require a frozen sample hash, held-out evaluation sample, both treatment arms, and explicit uncertainty limits.
3. Extract tree splits from the verified frozen policy object, assign stable breadth-first node letters, and translate only codes documented in the TEDS-D codebook.
4. Preserve Global (root-to-leaf), Local (one split and its branches), and Emergent (multi-node subtree) evidence scopes. Treat all three as hypothesis prompts, not confirmed moderators; a within-node effect is not a between-node moderation test.
5. Do not select findings only because a leaf has a small p-value.
6. Keep scientific mechanisms separate from treatment-selection, residual-confounding, and endpoint-recording threats.
7. Label policy-value jackknife standard errors as descriptive stability checks.


In [3]:
def read_rows(path):
    with path.open(newline='') as f: return list(csv.DictReader(f))

def policy_values(branch):
    rows=read_rows(run_dir/branch/'holdout_policy_value.csv')
    return {r['policy']:r for r in rows}

def loo_range(branch):
    rows=read_rows(run_dir/branch/'leave_one_state_out.csv')
    field='tree_minus_best_constant'
    vals=[float(r[field]) for r in rows if r.get(field) not in ('',None)]
    return [min(vals),max(vals)] if vals else None

TREE_LABELS={
 'RACE_4.0':('race','Black or African American'),
 'additional_substances_total_0.0':('additional substances','no additional substances recorded'),
 'additional_substances_total_nan':('additional substances','number of additional substances missing'),
 'ROUTE1_3.0':('primary route of administration','inhalation'),
 'ROUTE1_1.0':('primary route of administration','oral'),
 'PSOURCE_1.0':('referral source','individual, including self-referral'),
 'PSOURCE_2.0':('referral source','alcohol or drug use care provider'),
 'EMPLOY_4.0':('employment status','not in the labor force'),
 'SUB1_2.0':('primary substance','alcohol'),
 'SUB1_10.0':('primary substance','methamphetamine or speed'),
 'SUB1_5.0':('primary substance','heroin'),
}

def alpha_label(position):
    label=''
    while position>=0:
        position,remainder=divmod(position,26);label=chr(65+remainder)+label;position-=1
    return label
def node_letters(model):
    queue=[0];labels={}
    while queue:
        node=queue.pop(0);labels[node]=alpha_label(len(labels))
        left,right=int(model.children_left[node]),int(model.children_right[node])
        if left>=0:queue.extend([left,right])
    assert labels[0]=='A' and len(labels)==model.node_count and len(set(labels.values()))==model.node_count
    return labels

def tree_summary(branch):
    frozen=run_dir/branch/'frozen_policy.joblib'
    obj=joblib.load(frozen);model=obj['tree'].tree_model_.tree_;names=obj['feature_names'];letters=node_letters(model)
    leaf_inference={int(x['node']):x for x in results[branch]['leaf_inference']}
    node_inference={int(x['node']):x for x in results[branch]['node_inference']}
    splits=[];leaves=[];nodes=[]
    def walk(node,path,sequence,parent=None):
        left,right=int(model.children_left[node]),int(model.children_right[node]);row=node_inference[node]
        base={'node':node,'letter':letters[node],'parent_letter':letters[parent] if parent is not None else None,'node_sequence':' -> '.join(sequence+[letters[node]]),'rule':path,'heldout_n':row['n'],'heldout_states':row['state_clusters'],'effect_treatment_1_minus_0':row['effect']}
        if left<0:
            leaf=leaf_inference[node];record={**base,'node_type':'leaf','recommended_treatment_code':int(model.value[node,:,0].argmax()),'p_holm_within_tree':leaf['p_holm']}
            nodes.append(record);leaves.append(record);return
        raw=names[model.feature[node]];variable,category=TREE_LABELS[raw]
        split={**base,'node_type':'split','encoded_feature':raw,'variable':variable,'category':category,'left_letter':letters[left],'right_letter':letters[right]}
        nodes.append(split);splits.append(split)
        walk(left,path+[f'{variable}: not {category}'],sequence+[letters[node]],node)
        walk(right,path+[f'{variable}: {category}'],sequence+[letters[node]],node)
    walk(0,[],[])
    by_letter={x['letter']:x for x in nodes}
    globals_=[{'statement_id':f'G{i}','scope':'global_root_to_leaf','node_sequence':x['node_sequence'],'rule':x['rule'],'heldout_n':x['heldout_n'],'effect_treatment_1_minus_0':x['effect_treatment_1_minus_0']} for i,x in enumerate(leaves,1)]
    locals_=[]
    for i,x in enumerate(sorted(splits,key=lambda z:list(letters).index(z['node'])),1):
        left,right=by_letter[x['left_letter']],by_letter[x['right_letter']]
        locals_.append({'statement_id':f'L{i}','scope':'local_split_contrast','node_sequence':f"{x['letter']} -> {x['left_letter']} / {x['right_letter']}",'anchor_rule':x['rule'],'split':f"{x['variable']}: {x['category']}",'left_branch_effect':left['effect_treatment_1_minus_0'],'right_branch_effect':right['effect_treatment_1_minus_0'],'interpretation':'Descriptive branch contrast; not a formal test of between-branch moderation.'})
    emergent=[]
    for x in sorted(splits,key=lambda z:list(letters).index(z['node'])):
        prefix=x['node_sequence'];members=[leaf['letter'] for leaf in leaves if leaf['node_sequence'].startswith(prefix)]
        if len(members)>=2:emergent.append({'statement_id':f'E{len(emergent)+1}','scope':'emergent_node_group','anchor_letter':x['letter'],'node_letters':[x['letter'],*members],'node_sequence':x['letter']+' + '+ ' + '.join(members),'anchor_rule':x['rule'],'heldout_n':x['heldout_n'],'effect_treatment_1_minus_0':x['effect_treatment_1_minus_0'],'interpretation':'Multi-node subtree pattern; it may motivate a grouped hypothesis but does not establish a mechanism.'})
    assert len(globals_)==len(leaves) and len(locals_)==len(splits) and emergent
    return {'node_letter_order':'breadth-first beginning with A at the root','split_variables':sorted({x['variable'] for x in splits}),'nodes':nodes,'splits':splits,'leaves':leaves,'statements':{'global':globals_,'local':locals_,'emergent':emergent},
            'interpretation':'Global, Local, and Emergent statements preserve node-letter provenance. They motivate hypotheses but do not establish moderation or a causal mechanism.'}

res=results['residential_eligible']; out=results['outpatient_eligible']
rv=policy_values('residential_eligible'); ov=policy_values('outpatient_eligible')
res_effects=[x['effect'] for x in res['leaf_inference']]
assert max(res_effects)<0
assert float(rv['tree']['intensive_or_long_share'])==0
res_tree=tree_summary('residential_eligible'); out_tree=tree_summary('outpatient_eligible')

common_limits=[
 'Observational service setting; no-unmeasured-confounding cannot be verified.',
 'Recorded administrative completion is not a clinical recovery endpoint.',
 'Policy values target the overlap population and do not generalize automatically to all episodes.',
 'State-jackknife uncertainty is a descriptive stability check with few clusters.',
 'A policy-tree split motivates a subgroup hypothesis but does not establish effect moderation.'
]
evidence=[
 {
  'evidence_id':'residential_global_01','selection_status':'retrospective_exploratory',
  'analysis':'residential','treatment':'long-term versus short-term residential setting',
  'outcome':'recorded administrative completion (REASON=1)','target':'overlap-weighted population',
  'sample_membership_sha256':res['sample_membership_sha256'],'source_file_sha256':res['source_file_sha256'],
  'heldout_n':res['n_test'],'heldout_states':res['states_in_test'],
  'finding':f"The fitted policy assigns 0% of held-out episodes to long-term care. Estimated policy value equals always short-term ({float(rv['tree']['overlap_value']):.4f}) and exceeds always long-term ({float(rv['always_1']['overlap_value']):.4f}). All {len(res_effects)} held-out leaf effects are negative.",
  'tree_pattern':res_tree,
  'quantities':{'tree_value':float(rv['tree']['overlap_value']),'always_short_term_value':float(rv['always_0']['overlap_value']),'always_long_term_value':float(rv['always_1']['overlap_value']),'leaf_effect_min':min(res_effects),'leaf_effect_max':max(res_effects),'propensity_min':res['test_propensity_min'],'propensity_max':res['test_propensity_max'],'score_tail_percent':res['test_score_tail_percent']},
  'llm_task':'Develop concise substantive hypotheses about service-setting completion patterns and tree-defined heterogeneity. List treatment selection, residual confounding, and endpoint recording separately as validity threats.',
  'limits':common_limits+['The zero policy gain over always short-term is mechanical and does not validate the treatment contrast.','Extreme propensity scores and score tails require caution.']
 },
 {
  'evidence_id':'outpatient_policy_01','selection_status':'retrospective_exploratory',
  'analysis':'outpatient','treatment':'intensive versus non-intensive outpatient setting',
  'outcome':'recorded administrative completion (REASON=1)','target':'overlap-weighted population',
  'sample_membership_sha256':out['sample_membership_sha256'],'source_file_sha256':out['source_file_sha256'],
  'heldout_n':out['n_test'],'heldout_states':out['states_in_test'],
  'finding':f"The fitted policy value is {float(ov['tree']['overlap_value']):.4f}, compared with {float(ov['always_0']['overlap_value']):.4f} for always non-intensive and {float(ov['always_1']['overlap_value']):.4f} for always intensive. The descriptive gain over the better constant policy is {100*out['tree_minus_best_constant']:.3f} percentage points; its state-jackknife SE is {100*out['state_jackknife_se_tree_minus_always_0_descriptive']:.3f} points.",
  'tree_pattern':out_tree,
  'quantities':{'tree_value':float(ov['tree']['overlap_value']),'always_non_intensive_value':float(ov['always_0']['overlap_value']),'always_intensive_value':float(ov['always_1']['overlap_value']),'tree_minus_best_constant':out['tree_minus_best_constant'],'descriptive_state_jackknife_se':out['state_jackknife_se_tree_minus_always_0_descriptive'],'leave_one_state_out_gain_range':loo_range('outpatient_eligible'),'propensity_min':out['test_propensity_min'],'propensity_max':out['test_propensity_max'],'score_tail_percent':out['test_score_tail_percent']},
  'llm_task':'Develop concise substantive hypotheses about service-setting completion patterns and tree-defined heterogeneity. List treatment selection, residual confounding, and endpoint recording separately as validity threats.',
  'limits':common_limits+['Do not call the policy significantly superior; formal policy-value inference is not complete.','Completion and transfer coding varies across states; a 2023 state crosswalk is unavailable.']
 }
]
pending=[]
print(f'Selected {len(evidence)} evidence objects with verified tree structure.')
for x in evidence: print(x['analysis'],x['tree_pattern']['split_variables'])


Selected 2 evidence objects with verified tree structure.
residential ['additional substances', 'primary route of administration', 'race']
outpatient ['additional substances', 'employment status', 'primary substance', 'referral source']


In [4]:
stamp=datetime.now().astimezone().strftime('%Y%m%d_%H%M%S_%f')
out=ROOT/'S2/outputs'/f'S2_01_evidence_review_{stamp}'
out.mkdir(parents=True)
(out/'evidence_bundle.json').write_text(json.dumps({'status':'retrospective_exploratory','source_manifest':str(manifest_path.relative_to(ROOT)),'source_manifest_sha256':sha256(manifest_path),'evidence_objects':evidence,'pending_candidates':pending},indent=2))
with (out/'evidence_objects.csv').open('w',newline='') as f:
    fields=['evidence_id','analysis','treatment','outcome','target','heldout_n','heldout_states','finding','selection_status']
    w=csv.DictWriter(f,fieldnames=fields,extrasaction='ignore');w.writeheader();w.writerows(evidence)
files={p.name:sha256(p) for p in out.iterdir() if p.is_file()}
run_manifest={'status':'complete_retrospective_exploratory','created_at':datetime.now().astimezone().isoformat(),'source_manifest':str(manifest_path.relative_to(ROOT)),'source_manifest_sha256':sha256(manifest_path),'raw_episode_data_shared_with_llm':False,'selected_evidence_objects':len(evidence),'pending_candidates':len(pending),'files':files,'limits':['Evidence selection was frozen after Stage 1 results had been inspected.','Tree-defined subgroup hypotheses remain exploratory; tree splits do not establish moderation.','The bundle supports hypothesis generation, not causal or clinical recommendations.']}
(out/'manifest.json').write_text(json.dumps(run_manifest,indent=2))
print(json.dumps(run_manifest,indent=2))
print('Output:',out.relative_to(ROOT))

{
  "status": "complete_retrospective_exploratory",
  "created_at": "2026-09-27T14:28:55.248317-04:00",
  "source_manifest": "notebooks/code_0927/S1/outputs/S1_02_policy_trees_20260926_195240_351369/manifest.json",
  "source_manifest_sha256": "2eeebcd552967f1a360ad6a1d15af0d3082f98c5b5ed7064e2d6ec53dbe37882",
  "raw_episode_data_shared_with_llm": false,
  "selected_evidence_objects": 2,
  "pending_candidates": 0,
  "files": {
    "evidence_bundle.json": "46f10a477f47f3834d88484d237769ffa14bdb4aad618c7a013ed63306e84923",
    "evidence_objects.csv": "5c8abfc37b1d2dae5cb06565c85bf5375650b8787c9e81da0418e079e1e3416d"
  },
  "limits": [
    "Evidence selection was frozen after Stage 1 results had been inspected.",
    "Tree-defined subgroup hypotheses remain exploratory; tree splits do not establish moderation.",
    "The bundle supports hypothesis generation, not causal or clinical recommendations."
  ]
}
Output: notebooks/code_0927/S2/outputs/S2_01_evidence_review_20260927_142855_247162


## Interpretation and human review

Stage 3 develops standard scientific hypotheses from the two patterns and lists validity threats separately. Tree variables may define proposed subgroups, but the current trees do not confirm moderation. Before coauthor use, a researcher must confirm that each selected pattern and codebook label is accurate. Stage 4 evaluates the scientific hypotheses against retrieved literature and prepares independent expert review. A literature gap or previously unexamined subgroup is not labeled a Kuhnian anomaly.
